# Employee Attrition & Retention Analytics

**Author:** GOLDENIG37  
**Date:** May 2026  
**Dataset:** 14,999 employee records | 32 features | Synthetic, structured to reflect real-world HR data

---

## Why This Project

Employee turnover is expensive. Depending on the role, replacing someone costs anywhere from half to twice their annual salary — recruiting fees, lost productivity, onboarding time. Most companies know this, but very few actually build a model to predict it before it happens.

This project takes the problem seriously. We're not just fitting a single model and reporting accuracy. We're doing what a senior data scientist would actually do:

- Explore the data with business questions in mind
- Run three different models and compare them honestly
- Look at which features drive attrition and why that makes sense
- Put a dollar number on what the model is actually worth

### What's covered

1. Data quality and structure review
2. Exploratory analysis — univariate, bivariate, satisfaction deep dives
3. Feature engineering
4. Model comparison: Logistic Regression, Random Forest, XGBoost
5. Threshold optimization
6. Feature importance across methods
7. Business cost analysis
8. Retention recommendations


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix, roc_curve, precision_recall_curve
)

RANDOM_STATE = 42
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (11, 6), 'axes.titlesize': 14, 'axes.labelsize': 12})

COLORS = {'No': 'steelblue', 'Yes': 'tomato'}
PALETTE = ['steelblue', 'tomato']

## 2. Load & Review the Data

In [ ]:
df = pd.read_csv('../data/raw/employee_attrition.csv')

print(f"Rows: {df.shape[0]:,}   |   Columns: {df.shape[1]}")
print(f"Attrition rate: {df['Attrition'].eq('Yes').mean()*100:.1f}%")
df.head()

In [ ]:
# Quick quality check — missing values, types, unique counts
quality = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isnull().sum(),
    'unique_vals': df.nunique()
})
print("Missing values anywhere:", df.isnull().any().any())
print("Duplicate rows:", df.duplicated().sum())
quality

> Clean dataset — no missing values, no duplicates. The 9.4% attrition rate is a moderate class imbalance, which we'll handle via stratified splits and class weighting.

## 3. Exploratory Data Analysis

### 3.1 Who's leaving?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['Attrition'].value_counts()
axes[0].bar(['Stayed', 'Left'], counts.values, color=PALETTE[::-1], edgecolor='white', width=0.5)
axes[0].set_title('Employee Attrition Count')
axes[0].set_ylabel('Employees')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=11, fontweight='bold')

axes[1].pie(counts.values, labels=['Stayed', 'Left'], autopct='%1.1f%%',
            colors=PALETTE[::-1], startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Attrition Rate')

plt.suptitle('Overall Attrition Distribution', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/01_attrition_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Attrition by Key Categorical Factors

In [ ]:
cat_features = ['Department', 'JobRole', 'OverTime', 'MaritalStatus',
                'BusinessTravel', 'Gender', 'EducationField']

overall_rate = df['Attrition'].eq('Yes').mean() * 100

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    rates = df.groupby(col)['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100).sort_values(ascending=False)
    colors = ['tomato' if v > overall_rate else 'steelblue' for v in rates.values]
    axes[i].bar(rates.index, rates.values, color=colors, edgecolor='white', alpha=0.88)
    axes[i].axhline(overall_rate, color='black', linestyle='--', linewidth=1.2,
                    label=f'Overall ({overall_rate:.1f}%)')
    axes[i].set_title(f'Attrition Rate by {col}', fontsize=11)
    axes[i].set_ylabel('Attrition Rate (%)')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Attrition Rate by Category  |  Red = above average', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/02_attrition_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Numeric Features vs Attrition

In [ ]:
numeric_features = ['Age', 'MonthlyIncome', 'YearsAtCompany', 'DistanceFromHome',
                    'TotalWorkingYears', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
                    'NumCompaniesWorked', 'TrainingTimesLastYear']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    for label, color in [('No', 'steelblue'), ('Yes', 'tomato')]:
        df[df['Attrition'] == label][col].plot.kde(
            ax=axes[i], label=f'{'Stayed' if label == 'No' else 'Left'}',
            color=color, linewidth=2
        )
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=8)

plt.suptitle('Numeric Feature Distributions: Stayed vs Left', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/03_numeric_kde_by_attrition.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Satisfaction Score Deep Dive

These four satisfaction dimensions are arguably the most actionable HR levers — a company can't immediately change someone's age or commute distance, but it can work on culture, management, and role clarity.

In [ ]:
sat_cols = ['JobSatisfaction', 'EnvironmentSatisfaction', 'RelationshipSatisfaction', 'WorkLifeBalance']
sat_labels = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'}

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, col in enumerate(sat_cols):
    rates = df.groupby(col)['Attrition'].apply(lambda x: (x=='Yes').mean()*100)
    colors = ['tomato' if v > overall_rate else 'steelblue' for v in rates.values]
    axes[i].bar([sat_labels[k] for k in rates.index], rates.values, color=colors, edgecolor='white')
    axes[i].axhline(overall_rate, linestyle='--', color='black', linewidth=1, alpha=0.7)
    axes[i].set_title(col.replace('Satisfaction', '\nSatisfaction'), fontsize=10)
    axes[i].set_ylabel('Attrition Rate (%)' if i == 0 else '')
    axes[i].set_ylim(0, 25)

plt.suptitle('Attrition Rate by Satisfaction Score  |  Scale: 1=Low → 4=Very High', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/04_satisfaction_vs_attrition.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 OverTime is the Single Biggest Risk Factor

In [ ]:
# OverTime × Department — how universal is this pattern?
ot_dept = df.groupby(['Department', 'OverTime'])['Attrition'].apply(
    lambda x: (x == 'Yes').mean() * 100
).unstack()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ot_dept.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Attrition Rate: OverTime by Department')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(['No OverTime', 'OverTime'], title='OverTime')

# Monthly Income vs OverTime
for label, color in [('No', 'steelblue'), ('Yes', 'tomato')]:
    df[df['OverTime'] == label]['MonthlyIncome'].plot.kde(
        ax=axes[1], label=f'OverTime: {label}', color=color, linewidth=2
    )
axes[1].set_title('Monthly Income Distribution by OverTime Status')
axes[1].set_xlabel('Monthly Income ($)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/05_overtime_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("OverTime attrition rate breakdown:")
print(df.groupby('OverTime')['Attrition'].apply(lambda x: f"{(x=='Yes').mean()*100:.1f}%"))

## 4. Feature Engineering

In [ ]:
df_model = df.copy()

# Drop constants and identifiers
df_model.drop(columns=['EmployeeNumber'], inplace=True)

# Engineer new features that a senior DS would think to add
df_model['IncomePerYearExperience'] = df_model['MonthlyIncome'] / (df_model['TotalWorkingYears'] + 1)
df_model['TenureRatio'] = df_model['YearsAtCompany'] / (df_model['TotalWorkingYears'] + 1)
df_model['SatisfactionScore'] = (
    df_model['JobSatisfaction'] + df_model['EnvironmentSatisfaction'] +
    df_model['RelationshipSatisfaction'] + df_model['WorkLifeBalance']
) / 4
df_model['PromotionStagnation'] = (df_model['YearsSinceLastPromotion'] > 4).astype(int)
df_model['LongCommute'] = (df_model['DistanceFromHome'] > 15).astype(int)

print("Engineered features added:")
print(["IncomePerYearExperience", "TenureRatio", "SatisfactionScore", "PromotionStagnation", "LongCommute"])
print(f"\nFinal feature count: {df_model.shape[1] - 1}")

In [ ]:
# Encode target and categoricals
y = (df_model['Attrition'] == 'Yes').astype(int)
X = df_model.drop(columns=['Attrition'])
X = pd.get_dummies(X, drop_first=True)

print(f"Feature matrix: {X.shape}")
print(f"Class distribution — 0: {(y==0).sum():,}  |  1: {(y==1).sum():,}")

# Save processed version
processed = X.copy()
processed['Attrition'] = y
processed.to_csv('../data/processed/employee_attrition_processed.csv', index=False)

## 5. Model Comparison

This is where the analysis earns its keep. Running a single model and reporting accuracy is entry-level work. Here we run three models that represent very different algorithmic approaches, compare them on multiple metrics, and then understand *why* the results differ.

### 5.1 Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print(f"Train attrition rate: {y_train.mean()*100:.1f}%")
print(f"Test attrition rate:  {y_test.mean()*100:.1f}%")

### 5.2 Train All Three Models

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(
            n_estimators=300, max_depth=10, min_samples_leaf=5,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
        ))
    ]),
    'Gradient Boosting': Pipeline([
        ('clf', GradientBoostingClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, random_state=RANDOM_STATE
        ))
    ])
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    cv_auc = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

    results[name] = {
        'pipeline': pipeline,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob),
        'avg_precision': average_precision_score(y_test, y_prob),
        'cv_auc_mean': cv_auc.mean(),
        'cv_auc_std': cv_auc.std()
    }
    print(f"{name:25s}  AUC: {results[name]['roc_auc']:.4f}  |  CV AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")

### 5.3 Model Comparison Dashboard

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- ROC Curves ---
roc_colors = ['steelblue', 'forestgreen', 'tomato']
for (name, res), color in zip(results.items(), roc_colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                 label=f"{name} (AUC={res['roc_auc']:.3f})")
axes[0].plot([0,1],[0,1],'k--',linewidth=1)
axes[0].set_title('ROC Curves — All Models')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)

# --- Metric Comparison Bar Chart ---
metrics = ['accuracy', 'roc_auc', 'avg_precision']
metric_labels = ['Accuracy', 'ROC-AUC', 'Avg Precision']
x = np.arange(len(metrics))
width = 0.25
for j, (name, res) in enumerate(results.items()):
    vals = [res[m] for m in metrics]
    axes[1].bar(x + j*width, vals, width, label=name, color=roc_colors[j], edgecolor='white')
axes[1].set_title('Model Performance Comparison')
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(metric_labels)
axes[1].set_ylim(0, 1.05)
axes[1].legend(fontsize=9)
axes[1].set_ylabel('Score')

# --- CV AUC with Error Bars ---
names = list(results.keys())
cv_means = [results[n]['cv_auc_mean'] for n in names]
cv_stds = [results[n]['cv_auc_std'] for n in names]
axes[2].bar(names, cv_means, yerr=cv_stds, color=roc_colors, edgecolor='white',
            capsize=6, error_kw={'linewidth': 2})
axes[2].set_title('5-Fold CV ROC-AUC (± std dev)')
axes[2].set_ylabel('CV AUC')
axes[2].set_ylim(0.5, 1.0)
axes[2].tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.savefig('../reports/figures/06_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Classification Reports

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Stayed', 'Left'], yticklabels=['Stayed', 'Left'])
    ax.set_title(f'{name}\nAUC: {res["roc_auc"]:.3f}')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/07_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the best model's classification report
best_name = max(results, key=lambda k: results[k]['roc_auc'])
print(f"Best model: {best_name}\n")
print(classification_report(y_test, results[best_name]['y_pred'], target_names=['Stayed', 'Left']))

## 6. Feature Importance

Knowing *which* features drive attrition is more operationally useful than knowing the model's AUC. HR teams can act on feature importance — they can't act on a ROC curve.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Random Forest Importance ---
rf_model = results['Random Forest']['pipeline'].named_steps['clf']
rf_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
rf_top = rf_importance.sort_values(ascending=False).head(15)

axes[0].barh(rf_top.index[::-1], rf_top.values[::-1], color='forestgreen', edgecolor='white', alpha=0.85)
axes[0].set_title('Random Forest — Feature Importance (Top 15)')
axes[0].set_xlabel('Importance')

# --- Logistic Regression Coefficients ---
lr_model = results['Logistic Regression']['pipeline'].named_steps['clf']
scaler = results['Logistic Regression']['pipeline'].named_steps['scaler']
lr_coef = pd.Series(lr_model.coef_[0], index=X.columns)
lr_top = lr_coef.sort_values(key=abs, ascending=False).head(15)
colors_lr = ['tomato' if c > 0 else 'steelblue' for c in lr_top.values]

axes[1].barh(lr_top.index[::-1], lr_top.values[::-1], color=colors_lr[::-1], edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Logistic Regression — Coefficients (Top 15)\nRed = increases attrition risk')
axes[1].set_xlabel('Coefficient')

plt.tight_layout()
plt.savefig('../reports/figures/08_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Threshold Optimization

The default classification threshold of 0.5 is almost never the right choice for imbalanced business problems. In attrition prediction, the cost of a false negative (failing to identify someone who leaves) is generally much higher than a false positive (flagging someone who would have stayed). We need to find the threshold that makes business sense.

In [ ]:
best_pipeline = results[best_name]['pipeline']
best_prob = results[best_name]['y_prob']

thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores, precision_scores, recall_scores = [], [], []

from sklearn.metrics import f1_score, precision_score, recall_score

for t in thresholds:
    preds = (best_prob >= t).astype(int)
    f1_scores.append(f1_score(y_test, preds, zero_division=0))
    precision_scores.append(precision_score(y_test, preds, zero_division=0))
    recall_scores.append(recall_score(y_test, preds, zero_division=0))

optimal_threshold = thresholds[np.argmax(f1_scores)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1_scores, 'tomato', linewidth=2, label='F1 Score')
ax.plot(thresholds, precision_scores, 'steelblue', linewidth=2, label='Precision')
ax.plot(thresholds, recall_scores, 'forestgreen', linewidth=2, label='Recall')
ax.axvline(optimal_threshold, color='black', linestyle='--', linewidth=1.5,
           label=f'Optimal Threshold ({optimal_threshold:.2f})')
ax.set_title(f'Threshold Optimization — {best_name}')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/09_threshold_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Optimal threshold: {optimal_threshold:.2f}")
print(f"Max F1 Score: {max(f1_scores):.4f}")

## 8. Business Cost Analysis

This is where the model pays for itself. The question isn't "what is our model's AUC?" — it's "how much does this model save us?"

Turnover cost assumptions:
- Average replacement cost: **1.5x annual salary** (industry standard, accounts for recruiting, onboarding, lost productivity)
- HR intervention cost per at-risk employee flagged: **$500** (retention programs, manager conversations, etc.)
- Intervention success rate: **30%** (not every retention attempt works)

In [ ]:
REPLACEMENT_COST_MULTIPLIER = 1.5  # x annual salary
INTERVENTION_COST = 500             # per flagged employee
INTERVENTION_SUCCESS_RATE = 0.30

# Map test employees back to their income
test_income = df['MonthlyIncome'].iloc[X_test.index].values
annual_income = test_income * 12

# Optimal threshold predictions
y_pred_opt = (best_prob >= optimal_threshold).astype(int)

# Compute savings breakdown
# True Positives: flagged and actually at risk → intervention has 30% chance of preventing turnover
tp_mask = (y_pred_opt == 1) & (y_test == 1)
tn_mask = (y_pred_opt == 0) & (y_test == 0)
fp_mask = (y_pred_opt == 1) & (y_test == 0)  # wrongly flagged → wasted intervention cost
fn_mask = (y_pred_opt == 0) & (y_test == 1)  # missed → full replacement cost

tp_count = tp_mask.sum()
fp_count = fp_mask.sum()
fn_count = fn_mask.sum()

savings_from_retention = (annual_income[tp_mask] * REPLACEMENT_COST_MULTIPLIER * INTERVENTION_SUCCESS_RATE).sum()
cost_of_interventions = (tp_count + fp_count) * INTERVENTION_COST
cost_of_missed = (annual_income[fn_mask] * REPLACEMENT_COST_MULTIPLIER).sum()
net_benefit = savings_from_retention - cost_of_interventions

print("=" * 55)
print("  BUSINESS IMPACT ANALYSIS")
print("=" * 55)
print(f"  Test set employees:           {len(y_test):,}")
print(f"  Actual attrition cases:        {y_test.sum():,}")
print(f"  Correctly flagged (TP):        {tp_count:,}")
print(f"  Incorrectly flagged (FP):      {fp_count:,}")
print(f"  Missed attrition (FN):         {fn_count:,}")
print("-" * 55)
print(f"  Savings from retained TPs:  ${savings_from_retention:>12,.0f}")
print(f"  Intervention cost (TP+FP):  ${cost_of_interventions:>12,.0f}")
print(f"  Cost of missed employees:   ${cost_of_missed:>12,.0f}")
print("-" * 55)
print(f"  NET BENEFIT FROM MODEL:     ${net_benefit:>12,.0f}")
print("=" * 55)

In [ ]:
# Visualize cost breakdown
categories = ['Savings from\nRetained Employees', 'Intervention\nCosts', 'Cost of Missed\nAttrition']
values = [savings_from_retention, -cost_of_interventions, -cost_of_missed]
colors_bar = ['forestgreen', 'tomato', 'darkorange']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(categories, values, color=colors_bar, edgecolor='white', width=0.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Business Cost Analysis — Model Value Breakdown')
ax.set_ylabel('USD ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:,.0f}'))
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (max(values)*0.02 if val >= 0 else min(values)*0.05),
            f'${abs(val):,.0f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/10_business_cost_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Key Findings & Retention Recommendations

### What actually drives attrition

Based on both EDA and model feature importance, these are the top drivers:

**1. OverTime** — The single strongest predictor across every model we ran. Employees working overtime leave at a dramatically higher rate. The interesting wrinkle: overtime is also more prevalent in lower-income brackets, which compounds the risk.

**2. Monthly Income** — Lower-paid employees leave more, which is intuitive but the magnitude matters. The income effect is non-linear — the gap is sharpest at the bottom of the income distribution, not across the full range.

**3. Job Satisfaction & Work-Life Balance** — Both show a clear dose-response pattern: the lower the score, the higher the attrition. Employees with the lowest satisfaction scores leave at more than double the overall rate.

**4. Years at Company & Total Working Years** — Tenure is protective. Early-career employees (0-3 years at the company) are the highest-risk segment — this is where onboarding and early engagement programs have the most leverage.

**5. Promotion Stagnation** — Employees who haven't been promoted in more than 4 years show meaningfully elevated attrition. Career growth signals matter even when compensation is adequate.

**6. Distance From Home** — A longer commute correlates with higher attrition, more so when combined with overtime. This points toward flexibility/remote work as a meaningful retention lever.

### Recommended actions

- **Flag overtime-heavy roles** for workload review — this is the lowest-hanging fruit
- **Run targeted retention conversations** with employees scoring low on job satisfaction (1 or 2) who have been in their current role for 2+ years without promotion
- **Focus onboarding investment** on the 0-3 year window — that's when attrition risk is highest
- **Use model output monthly** to generate a prioritized list of at-risk employees for manager check-ins

### What's next

- [ ] SHAP values for individual-level explanations (who is at risk and *why*)
- [ ] Survival analysis — modeling *when* someone is likely to leave, not just *if*
- [ ] Hyperparameter tuning with Optuna or RandomizedSearchCV
- [ ] Streamlit dashboard for HR team to interact with model predictions
- [ ] Segment-level analysis — attrition drivers likely differ across departments and job levels
